In [0]:
# Databricks notebook cell

# %pip install requests pandas jsonschema
# %pip install openai>=1.56.0
# %pip install azure-keyvault-secrets azure-identity
# %pip install catboost scikit-learn pandas numpy
# %restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 88.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Databricks notebook cell

import json
import time
import math
import requests
import asyncio
import pandas as pd
from typing import Dict, Any, Optional, List, Awaitable
from openai import AsyncAzureOpenAI


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score, confusion_matrix, classification_report


In [0]:
API_KEY = 
client = AsyncAzureOpenAI(
                api_version="2024-10-21",
                azure_endpoint="https://oai-rtai-dev-aoai-east-001.openai.azure.com/",
                api_key=API_KEY,
            )

In [0]:
def get_system_prompt() -> str:
    return """
You are an expert in analyzing customer service call transcripts to evaluate agent performance based on the PCS rubric.

Your task is to extract PCS-related parameter values only.
Do NOT provide a final PCS score.

Use ONLY the transcript content provided.

Global rules:
1. Do not use outside knowledge.
2. Do not assume unsupported facts.
3. Evaluate only the original agent shown in the transcript.
4. A frustrated customer does not automatically imply poor agent performance.
5. If evidence is missing or unclear, use the designated unknown/default integer value.
6. Output valid JSON only.
7. No markdown, no prose, no explanation outside JSON.
8. Return JSON matching the schema exactly.

General coding rule:
- All PCS variables must be integers for preprocessing.
- Use 0 as the default unknown / not observed / cannot determine value unless a different mapping is explicitly defined below.
- Use evidence only from direct transcript wording or clearly observable interaction behavior.

Parameter definitions and integer mappings:

TRANSCRIPT META
- transcript_complete:
  1 = transcript appears complete enough to evaluate call handling
  0 = transcript appears partial, truncated, or incomplete

- multiple_agents_present:
  1 = more than one agent appears in the transcript
  0 = only one agent appears or unclear

- notes:
  Short plain-text note describing transcript quality, transfer context, ambiguity, or other extraction limitations.

CATEGORICAL PCS FEATURES AS INTEGERS
- agent_resolution_quality:
  0 = unknown / unclear
  1 = poor: no meaningful help, incorrect handling, or unresolved with weak handling
  2 = adequate: some help provided but limited, partial, or basic handling
  3 = good: appropriate help provided and issue reasonably addressed
  4 = excellent: thorough, effective, and highly competent resolution handling

- communication_quality:
  0 = unknown / unclear
  1 = unclear: confusing, disorganized, or hard to follow
  2 = acceptable: understandable but basic or inconsistent
  3 = clear: easy to follow, organized, and understandable
  4 = exceptional: especially clear, concise, and well-structured

- ownership_level:
  0 = unknown / unclear
  1 = low: avoids responsibility, gives minimal guidance, or leaves next steps vague
  2 = moderate: some responsibility shown but incomplete follow-through
  3 = strong: clearly takes responsibility, guides the issue, and supports next steps

- customer_effort_level:
  0 = unknown / unclear
  1 = high: customer had to repeat, push, clarify, or do extra work
  2 = medium: moderate effort required from the customer
  3 = low: agent made the interaction easy and minimized customer effort

BINARY / PRESENCE FEATURES
Use:
- 0 = no, not shown, unclear, or not applicable from transcript
- 1 = yes, clearly shown

- agent_greeted_professionally:
  Agent opens politely and appropriately.

- agent_showed_empathy:
  Agent expresses understanding, concern, care, or supportive language.

- agent_acknowledged_customer_emotion:
  Agent explicitly recognizes frustration, concern, inconvenience, or other customer emotion.

- agent_took_ownership:
  Agent takes responsibility for helping, guiding, or staying engaged with the issue.

- agent_apologized_when_appropriate:
  Agent apologizes when there is an issue, inconvenience, delay, or frustration warranting apology.

- agent_explained_next_steps:
  Agent states what will happen next or what actions should follow.

- agent_set_expectations:
  Agent gives timing, process expectations, limitations, or likely outcomes.

- agent_used_clear_language:
  Agent communicates in a direct, understandable, non-confusing way.

- agent_required_repeat_information:
  Customer had to repeat previously given information due to agent handling.

- agent_interrupted_customer:
  Agent cuts off or talks over the customer.

- agent_followed_process:
  Agent appears to follow an appropriate service process based on transcript evidence.

- agent_deescalated_effectively:
  Agent reduces tension or helps calm a difficult interaction.

- agent_escalated_or_transferred_appropriately:
  Agent escalates or transfers appropriately when needed and supported by the transcript.

- agent_closed_professionally:
  Agent ends the interaction politely and appropriately.

STRENGTH / LATENT SCORES
Use:
- 0 = unknown / cannot determine
- 1 = very weak
- 2 = weak
- 3 = moderate
- 4 = strong
- 5 = very strong

- agent_professionalism:
  Overall professionalism shown through tone, courtesy, and respectful handling.

- agent_empathy_strength:
  Overall strength of empathy and emotional acknowledgment.

- agent_ownership_strength:
  Overall strength of responsibility-taking and follow-through.

- process_adherence_strength:
  Overall strength of following appropriate support process.

- communication_clarity:
  Overall clarity, coherence, and ease of understanding of the agent’s communication.

- resolution_effectiveness:
  Overall effectiveness of the help or resolution provided by the agent.

- deescalation_effectiveness:
  Overall effectiveness of calming or stabilizing a tense interaction.

- overall_agent_impression_latent:
  Overall impression of the agent’s performance based only on transcript evidence.

EVIDENCE RULES
- Include short evidence quotes tied to extracted features.
- Use only direct transcript snippets or minimal exact quotations.
- Do not invent evidence.
- If no quote supports a feature, do not fabricate one.

CONFIDENCE
- feature_extraction_confidence:
  Float from 0.0 to 1.0 reflecting confidence in extraction quality.
  Use lower confidence when transcript is incomplete, ambiguous, or lacks direct evidence.

Return JSON matching the schema exactly.
""".strip()

In [0]:
# Databricks cell

def get_schema() -> Dict[str, Any]:
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "pcs_parameter_extraction",
            "schema": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "transcript_meta": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "transcript_complete": {"type": "boolean"},
                            "multiple_agents_present": {"type": "boolean"},
                            "notes": {"type": "string"}
                        },
                        "required": ["transcript_complete", "multiple_agents_present", "notes"]
                    },
                    "pcs_features": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "agent_resolution_quality": {
                                "type": "string",
                                "enum": ["poor", "adequate", "good", "excellent", "unclear", "unknown"]
                            },
                            "communication_quality": {
                                "type": "string",
                                "enum": ["unclear", "acceptable", "clear", "exceptional", "unknown"]
                            },
                            "ownership_level": {
                                "type": "string",
                                "enum": ["low", "moderate", "strong", "unknown"]
                            },
                            "customer_effort_level": {
                                "type": "string",
                                "enum": ["high", "medium", "low", "unknown"]
                            },

                            "agent_greeted_professionally": {"type": "integer"},
                            "agent_showed_empathy": {"type": "integer"},
                            "agent_acknowledged_customer_emotion": {"type": "integer"},
                            "agent_took_ownership": {"type": "integer"},
                            "agent_apologized_when_appropriate": {"type": "integer"},
                            "agent_explained_next_steps": {"type": "integer"},
                            "agent_set_expectations": {"type": "integer"},
                            "agent_used_clear_language": {"type": "integer"},
                            "agent_required_repeat_information": {"type": "integer"},
                            "agent_interrupted_customer": {"type": "integer"},
                            "agent_followed_process": {"type": "integer"},
                            "agent_deescalated_effectively": {"type": "integer"},
                            "agent_escalated_or_transferred_appropriately": {"type": "integer"},
                            "agent_closed_professionally": {"type": "integer"},

                            "agent_professionalism": {"type": ["integer", "null"]},
                            "agent_empathy_strength": {"type": ["integer", "null"]},
                            "agent_ownership_strength": {"type": ["integer", "null"]},
                            "process_adherence_strength": {"type": ["integer", "null"]},
                            "communication_clarity": {"type": ["integer", "null"]},
                            "resolution_effectiveness": {"type": ["integer", "null"]},
                            "deescalation_effectiveness": {"type": ["integer", "null"]},
                            "overall_agent_impression_latent": {"type": ["integer", "null"]}
                        },
                        "required": [
                            "agent_resolution_quality",
                            "communication_quality",
                            "ownership_level",
                            "customer_effort_level",
                            "agent_greeted_professionally",
                            "agent_showed_empathy",
                            "agent_acknowledged_customer_emotion",
                            "agent_took_ownership",
                            "agent_apologized_when_appropriate",
                            "agent_explained_next_steps",
                            "agent_set_expectations",
                            "agent_used_clear_language",
                            "agent_required_repeat_information",
                            "agent_interrupted_customer",
                            "agent_followed_process",
                            "agent_deescalated_effectively",
                            "agent_escalated_or_transferred_appropriately",
                            "agent_closed_professionally",
                            "agent_professionalism",
                            "agent_empathy_strength",
                            "agent_ownership_strength",
                            "process_adherence_strength",
                            "communication_clarity",
                            "resolution_effectiveness",
                            "deescalation_effectiveness",
                            "overall_agent_impression_latent"
                        ]
                    },
                    "evidence": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "properties": {
                                "feature": {"type": "string"},
                                "speaker": {"type": "string", "enum": ["customer", "agent", "unknown"]},
                                "quote": {"type": "string"}
                            },
                            "required": ["feature", "speaker", "quote"]
                        }
                    },
                    "confidence": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "feature_extraction_confidence": {"type": ["number", "null"]}
                        },
                        "required": ["feature_extraction_confidence"]
                    }
                },
                "required": ["transcript_meta", "pcs_features", "evidence", "confidence"]
            }
        }
    }

In [0]:
# Databricks cell

def build_user_prompt(transcript: str) -> str:
    return f"""
Extract PCS parameter values from the following call transcript.

Use only transcript evidence.
Return strict JSON only.

Transcript:
{transcript}
""".strip()

In [0]:
config = {
    "response_format": get_schema(),
    "system_prompt": get_system_prompt(),
    "reasoning_effort": "low",
    "model_name": "gpt-5-mini",
    "semaphore_value": 10,
    "n_retries": 30,
    "eval": False,
}

In [0]:
# Databricks cell

async def execute_completion(
    config: dict,
    user_input: str,
    client: AsyncAzureOpenAI = client
    ) -> tuple[str, str]:
    response_format = config["response_format"]
    system_prompt = config["system_prompt"]
    reasoning_effort = config["reasoning_effort"]
    model_name = config["model_name"]
    n_retries = config["n_retries"]

    last_exception = None

    for attempt in range(n_retries):
        try:
            response = await client.chat.completions.with_raw_response.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                response_format=response_format,
                reasoning_effort=reasoning_effort,
                model=model_name,
                timeout=60.0
            )

            headers = response.headers
            parsed = response.parse()
            content = parsed.choices[0].message.content

            debug_info = str({
                "rate_limits": {
                    "tokens_left": headers.get("x-ratelimit-remaining-tokens"),
                    "requests_left": headers.get("x-ratelimit-remaining-requests"),
                },
                "usage": {
                    "prompt_tokens": parsed.usage.prompt_tokens,
                    "completion_tokens": parsed.usage.completion_tokens,
                    "cached_tokens": getattr(parsed.usage.prompt_tokens_details, "cached_tokens", 0),
                    "total_tokens": parsed.usage.total_tokens,
                },
                "metadata": {
                    "model": parsed.model,
                    "finish_reason": parsed.choices[0].finish_reason,
                    "request_id": headers.get("x-request-id"),
                    "processing_ms": headers.get("openai-processing-ms")
                }
            })

            return content, debug_info

        except Exception as e:
            last_exception = e
            wait_seconds = min(2 ** attempt, 30)
            print(f"Attempt {attempt+1}/{n_retries} failed: {e}. Retrying in {wait_seconds}s")
            await asyncio.sleep(wait_seconds)

    fallback_content = json.dumps({
        "transcript_meta": {
            "transcript_complete": False,
            "multiple_agents_present": False,
            "notes": f"error: {str(last_exception)}"
        },
        "pcs_features": {
            "agent_resolution_quality": "unknown",
            "communication_quality": "unknown",
            "ownership_level": "unknown",
            "customer_effort_level": "unknown",
            "agent_greeted_professionally": 0,
            "agent_showed_empathy": 0,
            "agent_acknowledged_customer_emotion": 0,
            "agent_took_ownership": 0,
            "agent_apologized_when_appropriate": 0,
            "agent_explained_next_steps": 0,
            "agent_set_expectations": 0,
            "agent_used_clear_language": 0,
            "agent_required_repeat_information": 0,
            "agent_interrupted_customer": 0,
            "agent_followed_process": 0,
            "agent_deescalated_effectively": 0,
            "agent_escalated_or_transferred_appropriately": 0,
            "agent_closed_professionally": 0,
            "agent_professionalism": None,
            "agent_empathy_strength": None,
            "agent_ownership_strength": None,
            "process_adherence_strength": None,
            "communication_clarity": None,
            "resolution_effectiveness": None,
            "deescalation_effectiveness": None,
            "overall_agent_impression_latent": None
        },
        "evidence": [],
        "confidence": {
            "feature_extraction_confidence": 0.0
        }
    })

    fallback_debug_info = str({
        "rate_limits": {"tokens_left": 0, "requests_left": 0},
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "cached_tokens": 0, "total_tokens": 0},
        "metadata": {"model": model_name, "finish_reason": "error", "request_id": None, "processing_ms": None},
        "error": str(last_exception)
    })

    return fallback_content, fallback_debug_info

In [0]:
# Databricks cell

async def extract_pcs_parameters_for_row(
    row_id: int,
    transcript: str,
    config: dict,
    semaphore: asyncio.Semaphore,
    client: AsyncAzureOpenAI = client
) -> Dict[str, Any]:
    async with semaphore:
        content, debug_info = await execute_completion(
            config=config,
            user_input=build_user_prompt(transcript),
            client=client
        )

        try:
            parsed = json.loads(content)
            return {
                "row_id": row_id,
                "status": "ok",
                "llm_output": parsed,
                "debug_info": debug_info,
                "error_message": None
            }
        except Exception as e:
            return {
                "row_id": row_id,
                "status": "parse_error",
                "llm_output": None,
                "debug_info": debug_info,
                "error_message": str(e),
                "raw_content": content
            }

In [0]:
# Databricks cell

async def process_dataframe_async(
    df: pd.DataFrame,
    text_column: str,
    config: dict,
    client: AsyncAzureOpenAI = client
    ) -> List[Dict[str, Any]]:
    
    
    semaphore = asyncio.Semaphore(config["semaphore_value"])

    tasks = []
    for idx, row in df.iterrows():
        transcript = "" if pd.isna(row[text_column]) else str(row[text_column])
        tasks.append(
            extract_pcs_parameters_for_row(
                row_id=idx,
                transcript=transcript,
                config=config,
                semaphore=semaphore,
                client=client
            )
        )

    results = await asyncio.gather(*tasks, return_exceptions=False)
    return results

In [0]:
# Databricks cell

def flatten_llm_output(llm_output: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not llm_output:
        return {}

    flat = {}

    transcript_meta = llm_output.get("transcript_meta", {})
    for k, v in transcript_meta.items():
        flat[f"transcript_meta__{k}"] = v

    pcs_features = llm_output.get("pcs_features", {})
    for k, v in pcs_features.items():
        flat[f"pcs_features__{k}"] = v

    confidence = llm_output.get("confidence", {})
    for k, v in confidence.items():
        flat[f"confidence__{k}"] = v

    evidence = llm_output.get("evidence", [])
    flat["evidence_count"] = len(evidence)
    flat["evidence_json"] = json.dumps(evidence)

    return flat

In [0]:

input_csv_path = "./data/PCS_prepped2.csv"
output_csv_path = "./data/transcripts_with_pcs_parameters.csv"

df = pd.read_csv(input_csv_path)

assert "Text" in df.columns, "Input CSV must contain a 'Text' column"
print(f"Loaded {len(df)} rows")

results = await process_dataframe_async(
    df=df,
    text_column="Text",
    config=config,
    client=client
)

# Databricks cell

results_df = pd.DataFrame(results)

flattened_rows = []
for _, row in results_df.iterrows():
    flat = flatten_llm_output(row.get("llm_output"))
    flattened_rows.append(flat)

flat_df = pd.DataFrame(flattened_rows)

merged_df = pd.concat(
    [
        df.reset_index(drop=True),
        results_df[["status", "error_message", "debug_info"]].reset_index(drop=True),
        flat_df.reset_index(drop=True)
    ],
    axis=1
)

merged_df.head()

merged_df.to_csv(output_csv_path, index=False)
print(f"Saved output to: {output_csv_path}")

Loaded 38 rows
Saved output to: ./data/transcripts_with_pcs_parameters.csv
